# Automated VWAP Execution Research

This notebook is a separate execution-research layer for the VWAP Probability Band Engine.

It does **not** change the existing VWAP model, probability logic, visual overlay, live runner, or discretionary MT5 workflow.

The purpose is to research whether a rule-based execution layer can convert selected VWAP continuation setups into simulated trades.

Initial focus:

- continuation-only logic
- S-tier and A-tier setups only
- no B-tier discretionary/maybe trades
- no fresh orange entries
- no fresh red entries
- no reversal trades
- no chop/flat-overlay entries
- fixed Nasdaq point execution:
  - SL = -29 points
  - TP = +58 points
  - BE after +29 points

This notebook is research-only. It does not send orders to MT5.

## Workflow

The notebook will eventually follow this structure:

1. Import existing VWAP engine outputs.
2. Build automation-only features.
3. Classify continuation state.
4. Score trend health and band-shift quality.
5. Detect valid green reclaim / green rejection setups.
6. Simulate trade entry and management.
7. Export a trade log.
8. Review results and rejected setups.

Important principle:

The existing visual discretionary engine remains untouched.  
This notebook only adds an optional execution-research layer on top of the already-computed bands and context.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by walking upward until a .git folder is found.
    Falls back to the current working directory if .git is not found.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / ".git").exists():
            return path

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
LIVE_ARTIFACTS_DIR = PROJECT_ROOT / "live_artifacts"

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

## Automation Configuration

This config controls only the execution-research layer.

It does not alter:

- VWAP calculation
- sigma-band calculation
- probability calibration
- signal-generation internals
- live MT5 overlay behaviour
- existing discretionary workflow

In [ ]:
AUTO_CONFIG = {
    # Notebook mode
    "mode": "fully_automated_research",
    "research_only": True,

    # Scope
    "continuation_only": True,
    "allow_longs": True,
    "allow_shorts": True,
    "allow_reversals": False,
    "allow_b_tier": False,
    "allow_fresh_orange_entries": False,
    "allow_fresh_red_entries": False,

    # Setup tiers
    "s_tier_score": 5,
    "a_tier_score": 4,
    "minimum_tradeable_score": 4,

    # Execution
    "entry_mode": "next_bar_open",
    "one_trade_at_a_time": True,

    # Fixed Nasdaq point model
    # Distances are positive in config.
    # PnL result is signed later: SL = -29, TP = +58, BE = 0.
    "stop_loss_points": 29.0,
    "take_profit_points": 58.0,
    "breakeven_trigger_points": 29.0,

    # Risk controls
    "risk_per_trade_pct": 1.0,
    "max_consecutive_losses": 2,
    "daily_profit_cap_pct": 8.0,

    # Session controls
    "session_timezone": "Europe/London",
    "session_start": "14:00",
    "no_new_trades_after": "19:00",

    # Candle quality filters
    "min_body_ratio": 0.25,
    "min_close_through_green": 1.0,
    "max_extension_from_green": 8.0,

    # Shift / trend context
    "shift_lookback": 3,
    "vwap_cross_lookback": 8,
    "max_vwap_crosses_before_chop": 2,
}

AUTO_CONFIG

## Execution Sign Convention

The config stores point distances as positive numbers.

For trade PnL:

- stop loss = `-29`
- breakeven = `0`
- take profit = `+58`

For price levels:

Long trade:

- stop = entry - 29
- breakeven trigger = entry + 29
- target = entry + 58

Short trade:

- stop = entry + 29
- breakeven trigger = entry - 29
- target = entry - 58

In [ ]:
def build_trade_levels(entry_price: float, side: str, config: dict) -> dict:
    """
    Build fixed-point Nasdaq execution levels.

    Parameters
    ----------
    entry_price:
        Trade entry price.
    side:
        'long' or 'short'.
    config:
        AUTO_CONFIG dictionary.

    Returns
    -------
    dict
        Stop, breakeven trigger, and target prices.
    """
    side = side.lower()

    sl = config["stop_loss_points"]
    tp = config["take_profit_points"]
    be = config["breakeven_trigger_points"]

    if side == "long":
        return {
            "side": "long",
            "entry_price": entry_price,
            "stop_price": entry_price - sl,
            "breakeven_trigger_price": entry_price + be,
            "target_price": entry_price + tp,
        }

    if side == "short":
        return {
            "side": "short",
            "entry_price": entry_price,
            "stop_price": entry_price + sl,
            "breakeven_trigger_price": entry_price - be,
            "target_price": entry_price - tp,
        }

    raise ValueError("side must be 'long' or 'short'")

In [ ]:
# Quick sanity check

long_example = build_trade_levels(entry_price=20000.0, side="long", config=AUTO_CONFIG)
short_example = build_trade_levels(entry_price=20000.0, side="short", config=AUTO_CONFIG)

pd.DataFrame([long_example, short_example])

## Data Loading and Column Validation

This section prepares the automated execution notebook to consume existing VWAP engine outputs.

It does not recalculate the model.

The goal is to safely check that a DataFrame contains the fields needed for the automation layer:

- OHLC data
- timestamp
- VWAP/reference line
- upper/lower green bands
- upper/lower orange bands
- upper/lower red bands

Column aliases are supported because different exports may use slightly different names.

In [ ]:
REQUIRED_RAW_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
]

REQUIRED_ENGINE_OUTPUT_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
]

COLUMN_ALIASES = {
    "datetime": ["datetime", "time", "timestamp", "date", "Date", "Time", "Datetime"],
    "open": ["open", "Open", "OPEN"],
    "high": ["high", "High", "HIGH"],
    "low": ["low", "Low", "LOW"],
    "close": ["close", "Close", "CLOSE"],

    # Volume is useful later but not strictly required for execution rules
    "tick_volume": ["tick_volume", "volume", "Volume", "tickvol", "real_volume"],

    # VWAP/reference aliases
    "vwap": ["vwap", "VWAP", "reference", "ref", "reference_line"],

    # Upper bands
    "upper_green": [
        "upper_green", "upper_1", "upper_band_1", "band_1p", "band_1_plus",
        "band_1+", "1+", "z1_upper", "upper_sigma_1"
    ],
    "upper_orange": [
        "upper_orange", "upper_2", "upper_band_2", "band_2p", "band_2_plus",
        "band_2+", "2+", "z2_upper", "upper_sigma_2"
    ],
    "upper_red": [
        "upper_red", "upper_3", "upper_band_3", "band_3p", "band_3_plus",
        "band_3+", "3+", "z3_upper", "upper_sigma_3"
    ],

    # Lower bands
    "lower_green": [
        "lower_green", "lower_1", "lower_band_1", "band_1m", "band_1_minus",
        "band_1-", "1-", "z1_lower", "lower_sigma_1"
    ],
    "lower_orange": [
        "lower_orange", "lower_2", "lower_band_2", "band_2m", "band_2_minus",
        "band_2-", "2-", "z2_lower", "lower_sigma_2"
    ],
    "lower_red": [
        "lower_red", "lower_3", "lower_band_3", "band_3m", "band_3_minus",
        "band_3-", "3-", "z3_lower", "lower_sigma_3"
    ],
}

In [ ]:
def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Return the first matching column from a list of candidate names.
    """
    existing = set(df.columns)

    for col in candidates:
        if col in existing:
            return col

    # Case-insensitive fallback
    lower_map = {str(col).lower(): col for col in df.columns}
    for col in candidates:
        if str(col).lower() in lower_map:
            return lower_map[str(col).lower()]

    return None


def normalise_vwap_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename common VWAP/band export column names into the standard names used
    by the automation layer.

    This function does not change model values.
    It only standardises column names.
    """
    out = df.copy()
    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        matched_col = find_column(out, aliases)

        if matched_col is not None and matched_col != standard_name:
            rename_map[matched_col] = standard_name

    out = out.rename(columns=rename_map)

    return out

In [ ]:
def validate_automation_columns(df: pd.DataFrame, required_columns: list[str] | None = None) -> None:
    """
    Validate that the DataFrame has the columns required for automated
    VWAP execution research.

    Raises
    ------
    ValueError
        If required columns are missing.
    """
    if required_columns is None:
        required_columns = REQUIRED_AUTOMATION_COLUMNS

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required automation columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )

    print("All required automation columns are present.")

In [ ]:
def prepare_raw_ohlc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw MT5 OHLC data for later VWAP engine processing.

    This validates only the raw market data columns.

    It does not calculate VWAP.
    It does not calculate probability bands.
    It does not change the existing model logic.
    """
    out = normalise_vwap_columns(df)

    validate_automation_columns(out, required_columns=REQUIRED_RAW_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = ["open", "high", "low", "close"]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    # Optional MT5 fields
    for optional_col in ["tick_volume", "spread", "real_volume"]:
        if optional_col in out.columns:
            out[optional_col] = pd.to_numeric(out[optional_col], errors="coerce")

    # Basic candle features for later commits
    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out

def prepare_automation_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare an existing VWAP engine output DataFrame for automation research.

    This function expects VWAP and band columns to already exist.

    Use prepare_raw_ohlc_dataframe() for raw MT5 OHLC files.
    """
    out = normalise_vwap_columns(df)

    validate_automation_columns(out, required_columns=REQUIRED_ENGINE_OUTPUT_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out

In [ ]:
def list_candidate_data_files() -> list[Path]:
    """
    Recursively list likely CSV/Parquet files inside the project.

    This searches the whole repo so files are found even if they are not
    directly inside /data, /artifacts, or /live_artifacts.
    """
    patterns = ["*.csv", "*.parquet"]

    files = []
    for pattern in patterns:
        files.extend(PROJECT_ROOT.rglob(pattern))

    # Avoid common virtual environment/cache folders
    ignored_parts = {
        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
    }

    clean_files = [
        file for file in files
        if not any(part in ignored_parts for part in file.parts)
    ]

    return sorted(set(clean_files))


candidate_files = list_candidate_data_files()

print(f"Found {len(candidate_files)} candidate data files:")
for file in candidate_files[:50]:
    print(file.relative_to(PROJECT_ROOT))

## Optional: Load Existing Export

Use this section only when you have a CSV or Parquet file that already contains the VWAP/band output.

For now, this notebook does not generate new VWAP bands.  
It only prepares already-computed model output for automation research.

In [ ]:
# Default test file for automation research.
# Start with 30d because it is faster while building the notebook.

PREFERRED_DATA_FILE = "US100_cash_M1_NY_session_30d.csv"

candidate_files = list_candidate_data_files()

matching_files = [
    file for file in candidate_files
    if file.name == PREFERRED_DATA_FILE
]

if not matching_files:
    print(f"Could not find {PREFERRED_DATA_FILE}.")
    print("Available candidate files:")
    for file in candidate_files:
        print(file.relative_to(PROJECT_ROOT))
else:
    DATA_FILE = matching_files[0]
    print(f"Using file: {DATA_FILE.relative_to(PROJECT_ROOT)}")

    if DATA_FILE.suffix.lower() == ".csv":
        raw_df = pd.read_csv(DATA_FILE)
    elif DATA_FILE.suffix.lower() == ".parquet":
        raw_df = pd.read_parquet(DATA_FILE)
    else:
        raise ValueError(f"Unsupported file type: {DATA_FILE.suffix}")

    print("Raw columns:")
    print(list(raw_df.columns))

    raw_ohlc_df = prepare_raw_ohlc_dataframe(raw_df)

    print(f"\nLoaded {len(raw_ohlc_df):,} raw OHLC rows from {DATA_FILE.relative_to(PROJECT_ROOT)}")
    display(raw_ohlc_df.head())

## Existing VWAP Engine Integration

This section converts raw OHLC data into VWAP probability-band engine output using the existing `src/` modules.

It uses the existing project functions to compute:

- reference line
- sigma
- upper/lower bands
- z-score
- zone classification

Then it creates automation-friendly column names such as:

- `vwap`
- `upper_green`
- `upper_orange`
- `upper_red`
- `lower_green`
- `lower_orange`
- `lower_red`

In [ ]:
import sys
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config
importlib.reload(src.config)

from src.config import CONFIG
from src.reference import compute_reference
from src.sigma import compute_sigma, compute_bands
from src.zones import compute_zscore, classify_zones_series

print("Loaded existing VWAP engine modules.")
print(f"Reference type: {CONFIG['reference_type']}")
print(f"Vol method: {CONFIG['vol_method']}")
print(f"Zone thresholds: {CONFIG['zone_thresholds']}")

In [ ]:
def build_existing_engine_output(raw_ohlc_df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Build VWAP probability-band output from raw OHLC data using existing project logic.

    This function does not modify the model.
    It only calls the existing src functions and then adds automation-friendly aliases.
    """
    df = raw_ohlc_df.copy()

    # Required by existing reference calculation
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    if "tick_volume" not in df.columns:
        df["tick_volume"] = 1.0

    df["tick_volume"] = pd.to_numeric(df["tick_volume"], errors="coerce").fillna(1.0).clip(lower=1.0)

    df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3.0
    df["session_date"] = df["datetime"].dt.date

    # Existing model logic
    df["reference"] = compute_reference(df, config)
    df["price_deviation"] = df["close"] - df["reference"]

    df["sigma"] = compute_sigma(df, config)

    bands = compute_bands(df, df["sigma"])
    df = pd.concat([df, bands], axis=1)

    df["z_score"] = compute_zscore(df)
    df["zone"] = classify_zones_series(df["z_score"], config["zone_thresholds"])

    # Automation-friendly aliases
    df["vwap"] = df["reference"]

    df["upper_green"] = df["band_1p"]
    df["upper_orange"] = df["band_2p"]
    df["upper_red"] = df["band_3p"]

    df["lower_green"] = df["band_1n"]
    df["lower_orange"] = df["band_2n"]
    df["lower_red"] = df["band_3n"]

    # Keep candle features consistent
    df["candle_range"] = df["high"] - df["low"]
    df["candle_body"] = (df["close"] - df["open"]).abs()
    df["body_ratio"] = np.where(
        df["candle_range"] > 0,
        df["candle_body"] / df["candle_range"],
        0.0,
    )

    return df

In [ ]:
engine_df = build_existing_engine_output(raw_ohlc_df, CONFIG)

print(f"Built existing engine output for {len(engine_df):,} rows.")

display_cols = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "sigma",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

display(engine_df[display_cols].tail())

In [ ]:
automation_ready_df = prepare_automation_dataframe(engine_df)

print(f"Automation-ready DataFrame: {len(automation_ready_df):,} rows")
print("Required automation columns are now available.")

display(
    automation_ready_df[
        [
            "datetime",
            "close",
            "vwap",
            "upper_green",
            "upper_orange",
            "upper_red",
            "lower_green",
            "lower_orange",
            "lower_red",
            "body_ratio",
        ]
    ].tail()
)

## Derived Automation Features

This section creates automation-only features from the existing VWAP engine output.

It does not change:

- VWAP calculation
- band calculation
- probability model
- visual discretionary overlay
- live MT5 workflow

These features are used only by the automated execution research layer.

The goal is to measure:

- band-shift strength
- red-band trend aggression
- acceptance above/below VWAP
- acceptance inside the trend lane
- trend damage
- VWAP chop/crossing
- band compression

In [ ]:
AUTO_CONFIG.update(
    {
        # Feature lookbacks
        "acceptance_lookback": 3,
        "trend_lane_lookback": 5,
        "trend_damage_lookback": 5,
        "compression_lookback": 5,
        "flat_vwap_lookback": 5,

        # Chop / compression settings
        "flat_vwap_threshold_points": 3.0,
        "min_band_expansion_points": 0.0,

        # Red-band shift strength buckets over shift_lookback
        "red_shift_minimum_points": 3.0,
        "red_shift_good_points": 5.0,
        "red_shift_strong_points": 7.0,
        "red_shift_very_strong_points": 10.0,
        "red_shift_extreme_points": 20.0,
        "red_shift_abnormal_points": 40.0,
    }
)

AUTO_CONFIG

In [ ]:
def consecutive_true_count(condition: pd.Series) -> pd.Series:
    """
    Count consecutive True values.

    Example:
    False, True, True, False, True -> 0, 1, 2, 0, 1
    """
    condition = condition.fillna(False).astype(bool)

    counts = []
    current_count = 0

    for value in condition:
        if value:
            current_count += 1
        else:
            current_count = 0

        counts.append(current_count)

    return pd.Series(counts, index=condition.index)


def classify_red_shift_strength(value: float, config: dict) -> str:
    """
    Classify red-band shift strength in Nasdaq points.

    The input should be positive and already converted into trend-direction strength.
    """
    if pd.isna(value):
        return "unknown"

    if value < config["red_shift_minimum_points"]:
        return "weak"

    if value < config["red_shift_good_points"]:
        return "minimum"

    if value < config["red_shift_strong_points"]:
        return "good"

    if value < config["red_shift_very_strong_points"]:
        return "strong"

    if value < config["red_shift_extreme_points"]:
        return "very_strong"

    if value < config["red_shift_abnormal_points"]:
        return "extreme"

    return "abnormal_news_or_crash_regime"

In [ ]:
def add_automation_features(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add automation-only derived features to an existing VWAP engine DataFrame.

    This function does not alter the existing model logic.
    It only derives extra research columns from already-computed VWAP/band output.
    """
    out = df.copy()
    out = out.sort_values("datetime").reset_index(drop=True)

    shift_lookback = config["shift_lookback"]
    acceptance_lookback = config["acceptance_lookback"]
    trend_lane_lookback = config["trend_lane_lookback"]
    trend_damage_lookback = config["trend_damage_lookback"]
    compression_lookback = config["compression_lookback"]
    flat_vwap_lookback = config["flat_vwap_lookback"]
    vwap_cross_lookback = config["vwap_cross_lookback"]

    # ------------------------------------------------------------------
    # Band shifts
    # ------------------------------------------------------------------
    shift_columns = [
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    for col in shift_columns:
        out[f"{col}_shift"] = out[col] - out[col].shift(shift_lookback)

    # Directional red-band strength.
    # For bullish continuation, upper red shifting up is strength.
    # For bearish continuation, lower red shifting down is strength, so multiply by -1.
    out["bullish_red_shift_strength"] = out["upper_red_shift"]
    out["bearish_red_shift_strength"] = -out["lower_red_shift"]

    out["bullish_red_shift_label"] = out["bullish_red_shift_strength"].apply(
        lambda value: classify_red_shift_strength(value, config)
    )

    out["bearish_red_shift_label"] = out["bearish_red_shift_strength"].apply(
        lambda value: classify_red_shift_strength(value, config)
    )

    # ------------------------------------------------------------------
    # Band expansion / compression
    # ------------------------------------------------------------------
    out["green_band_width"] = out["upper_green"] - out["lower_green"]
    out["orange_band_width"] = out["upper_orange"] - out["lower_orange"]
    out["red_band_width"] = out["upper_red"] - out["lower_red"]

    out["green_band_width_change"] = out["green_band_width"] - out["green_band_width"].shift(compression_lookback)
    out["orange_band_width_change"] = out["orange_band_width"] - out["orange_band_width"].shift(compression_lookback)
    out["red_band_width_change"] = out["red_band_width"] - out["red_band_width"].shift(compression_lookback)

    out["bands_expanding"] = out["red_band_width_change"] > config["min_band_expansion_points"]
    out["bands_compressing"] = out["red_band_width_change"] < 0

    # Opposite-side expansion context.
    # Bullish trend quality improves when lower bands shift down/away while upper bands shift up.
    # Bearish trend quality improves when upper bands shift up/away while lower bands shift down.
    out["bullish_opposite_band_expansion"] = -out["lower_red_shift"]
    out["bearish_opposite_band_expansion"] = out["upper_red_shift"]

    # ------------------------------------------------------------------
    # VWAP acceptance
    # ------------------------------------------------------------------
    out["close_above_vwap"] = out["close"] > out["vwap"]
    out["close_below_vwap"] = out["close"] < out["vwap"]

    out["closes_above_vwap_count"] = (
        out["close_above_vwap"]
        .astype(int)
        .rolling(acceptance_lookback, min_periods=1)
        .sum()
    )

    out["closes_below_vwap_count"] = (
        out["close_below_vwap"]
        .astype(int)
        .rolling(acceptance_lookback, min_periods=1)
        .sum()
    )

    out["accepted_above_vwap"] = out["closes_above_vwap_count"] >= 2
    out["accepted_below_vwap"] = out["closes_below_vwap_count"] >= 2

    # ------------------------------------------------------------------
    # Trend-lane acceptance
    # ------------------------------------------------------------------
    # Bullish lane = between upper green and upper orange.
    # Bearish lane = between lower orange and lower green.
    out["close_in_bullish_green_lane"] = (
        (out["close"] >= out["upper_green"])
        & (out["close"] <= out["upper_orange"])
    )

    out["close_in_bearish_green_lane"] = (
        (out["close"] <= out["lower_green"])
        & (out["close"] >= out["lower_orange"])
    )

    out["bullish_lane_close_count"] = (
        out["close_in_bullish_green_lane"]
        .astype(int)
        .rolling(trend_lane_lookback, min_periods=1)
        .sum()
    )

    out["bearish_lane_close_count"] = (
        out["close_in_bearish_green_lane"]
        .astype(int)
        .rolling(trend_lane_lookback, min_periods=1)
        .sum()
    )

    # ------------------------------------------------------------------
    # Trend damage / trend death
    # ------------------------------------------------------------------
    out["close_below_upper_green"] = out["close"] < out["upper_green"]
    out["close_above_lower_green"] = out["close"] > out["lower_green"]

    out["consecutive_closes_below_upper_green"] = consecutive_true_count(out["close_below_upper_green"])
    out["consecutive_closes_above_lower_green"] = consecutive_true_count(out["close_above_lower_green"])

    out["bullish_trend_dead_by_green_loss"] = (
        out["consecutive_closes_below_upper_green"] >= trend_damage_lookback
    )

    out["bearish_trend_dead_by_green_loss"] = (
        out["consecutive_closes_above_lower_green"] >= trend_damage_lookback
    )

    # Second close back inside green after temporary loss.
    # Bullish: price was below/at upper green, then closes back above upper green for 2 candles.
    # Bearish: price was above/at lower green, then closes back below lower green for 2 candles.
    out["bullish_second_close_back_above_green"] = (
        (out["close"] > out["upper_green"])
        & (out["close"].shift(1) > out["upper_green"].shift(1))
        & (out["close"].shift(2) <= out["upper_green"].shift(2))
    )

    out["bearish_second_close_back_below_green"] = (
        (out["close"] < out["lower_green"])
        & (out["close"].shift(1) < out["lower_green"].shift(1))
        & (out["close"].shift(2) >= out["lower_green"].shift(2))
    )

    # ------------------------------------------------------------------
    # VWAP crossing / chop markers
    # ------------------------------------------------------------------
    out["vwap_side"] = np.where(
        out["close"] > out["vwap"],
        1,
        np.where(out["close"] < out["vwap"], -1, 0),
    )

    out["vwap_cross"] = (
        (out["vwap_side"] != out["vwap_side"].shift(1))
        & (out["vwap_side"] != 0)
        & (out["vwap_side"].shift(1) != 0)
    )

    out["vwap_cross_count"] = (
        out["vwap_cross"]
        .astype(int)
        .rolling(vwap_cross_lookback, min_periods=1)
        .sum()
    )

    out["vwap_shift_flat_check"] = out["vwap"] - out["vwap"].shift(flat_vwap_lookback)
    out["vwap_is_flat"] = out["vwap_shift_flat_check"].abs() <= config["flat_vwap_threshold_points"]

    out["possible_chop"] = (
        (out["vwap_cross_count"] >= config["max_vwap_crosses_before_chop"])
        & out["vwap_is_flat"]
    ) | (
        out["bands_compressing"]
        & out["vwap_is_flat"]
    )

    return out

In [ ]:
features_df = add_automation_features(automation_ready_df, AUTO_CONFIG)

print(f"Feature DataFrame: {len(features_df):,} rows")
print(f"Columns added: {len(features_df.columns) - len(automation_ready_df.columns)}")

preview_cols = [
    "datetime",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "vwap_shift",
    "upper_red_shift",
    "lower_red_shift",
    "bullish_red_shift_strength",
    "bearish_red_shift_strength",
    "bullish_red_shift_label",
    "bearish_red_shift_label",
    "accepted_above_vwap",
    "accepted_below_vwap",
    "bullish_lane_close_count",
    "bearish_lane_close_count",
    "vwap_cross_count",
    "bands_compressing",
    "possible_chop",
]

display(features_df[preview_cols].tail(20))

In [ ]:
summary = {
    "rows": len(features_df),
    "possible_chop_rows": int(features_df["possible_chop"].sum()),
    "accepted_above_vwap_rows": int(features_df["accepted_above_vwap"].sum()),
    "accepted_below_vwap_rows": int(features_df["accepted_below_vwap"].sum()),
    "bullish_second_close_back_above_green": int(features_df["bullish_second_close_back_above_green"].sum()),
    "bearish_second_close_back_below_green": int(features_df["bearish_second_close_back_below_green"].sum()),
    "bullish_trend_dead_by_green_loss": int(features_df["bullish_trend_dead_by_green_loss"].sum()),
    "bearish_trend_dead_by_green_loss": int(features_df["bearish_trend_dead_by_green_loss"].sum()),
}

pd.Series(summary)

This section only creates automation-derived context features.

It does not yet create:

- long signals
- short signals
- S/A/B setup classification
- entries
- stops
- take-profits
- breakeven logic
- trade logs

The next commit will use these features to build a continuation-state classifier:

- bullish continuation candidate
- bearish continuation candidate
- chop / unclear
- extension / stretched
- invalid / no trade

## Simple Green Reclaim / Rejection Signals

This is the first MVP automation step.

The goal is not to capture every discretionary nuance yet.

This section only detects simple mechanical continuation setups:

Long green reclaim:

- accepted above VWAP
- close above VWAP
- candle touches/dips into upper green
- candle closes back above upper green
- candle body quality is acceptable
- close is not already at/above upper orange
- close is not too far above upper green

Short lower-green rejection:

- accepted below VWAP
- close below VWAP
- candle touches/pokes into lower green
- candle closes back below lower green
- candle body quality is acceptable
- close is not already at/below lower orange
- close is not too far below lower green

This section does not simulate trades yet.

In [ ]:
def add_simple_green_signals(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Add simple MVP green reclaim / rejection signals.

    This is deliberately simple.

    It does not:
    - simulate trades
    - apply daily stop rules
    - apply breakeven logic
    - classify S/A/B setup tiers
    - place orders
    """
    out = df.copy()

    min_body_ratio = config["min_body_ratio"]
    min_close_through_green = config["min_close_through_green"]
    max_extension_from_green = config["max_extension_from_green"]

    # ------------------------------------------------------------
    # Long: upper-green reclaim
    # ------------------------------------------------------------
    out["long_touched_upper_green"] = out["low"] <= out["upper_green"]
    out["long_closed_above_upper_green"] = out["close"] > out["upper_green"]

    out["long_close_through_green_points"] = out["close"] - out["upper_green"]
    out["long_extension_from_green_points"] = out["close"] - out["upper_green"]

    out["long_close_through_green_valid"] = (
        out["long_close_through_green_points"] >= min_close_through_green
    )

    out["long_extension_valid"] = (
        out["long_extension_from_green_points"] <= max_extension_from_green
    )

    out["long_body_valid"] = out["body_ratio"] >= min_body_ratio

    out["long_not_orange_chase"] = out["close"] < out["upper_orange"]

    out["simple_long_green_reclaim"] = (
        out["accepted_above_vwap"]
        & (out["close"] > out["vwap"])
        & out["long_touched_upper_green"]
        & out["long_closed_above_upper_green"]
        & out["long_close_through_green_valid"]
        & out["long_extension_valid"]
        & out["long_body_valid"]
        & out["long_not_orange_chase"]
        & ~out["possible_chop"]
    )

    # ------------------------------------------------------------
    # Short: lower-green rejection
    # ------------------------------------------------------------
    out["short_touched_lower_green"] = out["high"] >= out["lower_green"]
    out["short_closed_below_lower_green"] = out["close"] < out["lower_green"]

    out["short_close_through_green_points"] = out["lower_green"] - out["close"]
    out["short_extension_from_green_points"] = out["lower_green"] - out["close"]

    out["short_close_through_green_valid"] = (
        out["short_close_through_green_points"] >= min_close_through_green
    )

    out["short_extension_valid"] = (
        out["short_extension_from_green_points"] <= max_extension_from_green
    )

    out["short_body_valid"] = out["body_ratio"] >= min_body_ratio

    out["short_not_orange_chase"] = out["close"] > out["lower_orange"]

    out["simple_short_green_rejection"] = (
        out["accepted_below_vwap"]
        & (out["close"] < out["vwap"])
        & out["short_touched_lower_green"]
        & out["short_closed_below_lower_green"]
        & out["short_close_through_green_valid"]
        & out["short_extension_valid"]
        & out["short_body_valid"]
        & out["short_not_orange_chase"]
        & ~out["possible_chop"]
    )

    # ------------------------------------------------------------
    # Combined signal label
    # ------------------------------------------------------------
    out["simple_signal_side"] = np.select(
        [
            out["simple_long_green_reclaim"],
            out["simple_short_green_rejection"],
        ],
        [
            "long",
            "short",
        ],
        default="none",
    )

    out["simple_signal_name"] = np.select(
        [
            out["simple_long_green_reclaim"],
            out["simple_short_green_rejection"],
        ],
        [
            "LONG_GREEN_RECLAIM",
            "SHORT_GREEN_REJECTION",
        ],
        default="NO_SIGNAL",
    )

    return out

In [ ]:
signals_df = add_simple_green_signals(features_df, AUTO_CONFIG)

signal_counts = signals_df["simple_signal_name"].value_counts(dropna=False)
display(signal_counts)

signal_rows = signals_df[signals_df["simple_signal_name"] != "NO_SIGNAL"].copy()

print(f"Simple signals found: {len(signal_rows):,}")

display(
    signal_rows[
        [
            "datetime",
            "simple_signal_name",
            "simple_signal_side",
            "open",
            "high",
            "low",
            "close",
            "vwap",
            "upper_green",
            "upper_orange",
            "lower_green",
            "lower_orange",
            "body_ratio",
            "accepted_above_vwap",
            "accepted_below_vwap",
            "possible_chop",
            "long_close_through_green_points",
            "long_extension_from_green_points",
            "short_close_through_green_points",
            "short_extension_from_green_points",
        ]
    ].tail(50)
)

In [ ]:
# Compact semi-automated style preview

signal_preview = signal_rows.copy()

signal_preview["entry_plan"] = "next_bar_open"
signal_preview["stop_points"] = -AUTO_CONFIG["stop_loss_points"]
signal_preview["breakeven_after_points"] = AUTO_CONFIG["breakeven_trigger_points"]
signal_preview["target_points"] = AUTO_CONFIG["take_profit_points"]

display(
    signal_preview[
        [
            "datetime",
            "simple_signal_name",
            "simple_signal_side",
            "entry_plan",
            "stop_points",
            "breakeven_after_points",
            "target_points",
            "close",
            "vwap",
            "body_ratio",
        ]
    ].tail(50)
)

## Fixed-Point Trade Simulation

This section turns simple green reclaim / rejection signals into simulated trades.

Rules:

- Entry is at the next bar open after the signal candle.
- Long stop = entry - 29 points.
- Long breakeven trigger = entry + 29 points.
- Long target = entry + 58 points.
- Short stop = entry + 29 points.
- Short breakeven trigger = entry - 29 points.
- Short target = entry - 58 points.
- Only one trade can be open at a time.

Intrabar ambiguity rule:

If stop and target are both touched inside the same candle, the simulator assumes the worse outcome first. This makes the first backtest conservative.

In [ ]:
def normalise_timestamp_to_session_time(timestamp, config: dict):
    """
    Convert timestamp to the configured session timezone.
    """
    ts = pd.Timestamp(timestamp)

    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")

    return ts.tz_convert(config["session_timezone"])


def is_before_no_new_trades_cutoff(timestamp, config: dict) -> bool:
    """
    Check whether a new trade is allowed based on the configured cutoff time.
    """
    if "no_new_trades_after" not in config or config["no_new_trades_after"] is None:
        return True

    session_ts = normalise_timestamp_to_session_time(timestamp, config)

    cutoff = pd.to_datetime(config["no_new_trades_after"]).time()

    return session_ts.time() < cutoff

In [ ]:
def simulate_single_trade(
    df: pd.DataFrame,
    entry_idx: int,
    signal_idx: int,
    side: str,
    signal_name: str,
    config: dict,
) -> dict:
    """
    Simulate one fixed-point trade from entry_idx onward.

    Uses conservative intrabar assumptions.
    """
    entry_row = df.iloc[entry_idx]
    signal_row = df.iloc[signal_idx]

    entry_price = float(entry_row["open"])
    levels = build_trade_levels(entry_price, side, config)

    stop_price = levels["stop_price"]
    be_trigger_price = levels["breakeven_trigger_price"]
    target_price = levels["target_price"]

    be_active = False

    for j in range(entry_idx, len(df)):
        row = df.iloc[j]

        high = float(row["high"])
        low = float(row["low"])

        if side == "long":
            if not be_active:
                original_sl_hit = low <= stop_price
                target_hit = high >= target_price
                be_hit = high >= be_trigger_price

                # Conservative ambiguity: SL first if both sides touch.
                if original_sl_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": stop_price,
                        "exit_reason": "SL",
                        "points_result": -config["stop_loss_points"],
                        "R_result": -1.0,
                        "bars_held": j - entry_idx + 1,
                    }

                if target_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": target_price,
                        "exit_reason": "TP",
                        "points_result": config["take_profit_points"],
                        "R_result": config["take_profit_points"] / config["stop_loss_points"],
                        "bars_held": j - entry_idx + 1,
                    }

                if be_hit:
                    be_active = True

                    # Conservative same-candle BE check.
                    if low <= entry_price:
                        return {
                            "signal_idx": signal_idx,
                            "entry_idx": entry_idx,
                            "exit_idx": j,
                            "signal_time": signal_row["datetime"],
                            "entry_time": entry_row["datetime"],
                            "exit_time": row["datetime"],
                            "side": side,
                            "signal_name": signal_name,
                            "entry_price": entry_price,
                            "stop_price_initial": stop_price,
                            "be_trigger_price": be_trigger_price,
                            "target_price": target_price,
                            "exit_price": entry_price,
                            "exit_reason": "BE",
                            "points_result": 0.0,
                            "R_result": 0.0,
                            "bars_held": j - entry_idx + 1,
                        }

            else:
                be_stop_hit = low <= entry_price
                target_hit = high >= target_price

                # Conservative ambiguity after BE: BE first.
                if be_stop_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": entry_price,
                        "exit_reason": "BE",
                        "points_result": 0.0,
                        "R_result": 0.0,
                        "bars_held": j - entry_idx + 1,
                    }

                if target_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": target_price,
                        "exit_reason": "TP",
                        "points_result": config["take_profit_points"],
                        "R_result": config["take_profit_points"] / config["stop_loss_points"],
                        "bars_held": j - entry_idx + 1,
                    }

        elif side == "short":
            if not be_active:
                original_sl_hit = high >= stop_price
                target_hit = low <= target_price
                be_hit = low <= be_trigger_price

                # Conservative ambiguity: SL first if both sides touch.
                if original_sl_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": stop_price,
                        "exit_reason": "SL",
                        "points_result": -config["stop_loss_points"],
                        "R_result": -1.0,
                        "bars_held": j - entry_idx + 1,
                    }

                if target_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": target_price,
                        "exit_reason": "TP",
                        "points_result": config["take_profit_points"],
                        "R_result": config["take_profit_points"] / config["stop_loss_points"],
                        "bars_held": j - entry_idx + 1,
                    }

                if be_hit:
                    be_active = True

                    # Conservative same-candle BE check.
                    if high >= entry_price:
                        return {
                            "signal_idx": signal_idx,
                            "entry_idx": entry_idx,
                            "exit_idx": j,
                            "signal_time": signal_row["datetime"],
                            "entry_time": entry_row["datetime"],
                            "exit_time": row["datetime"],
                            "side": side,
                            "signal_name": signal_name,
                            "entry_price": entry_price,
                            "stop_price_initial": stop_price,
                            "be_trigger_price": be_trigger_price,
                            "target_price": target_price,
                            "exit_price": entry_price,
                            "exit_reason": "BE",
                            "points_result": 0.0,
                            "R_result": 0.0,
                            "bars_held": j - entry_idx + 1,
                        }

            else:
                be_stop_hit = high >= entry_price
                target_hit = low <= target_price

                # Conservative ambiguity after BE: BE first.
                if be_stop_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": entry_price,
                        "exit_reason": "BE",
                        "points_result": 0.0,
                        "R_result": 0.0,
                        "bars_held": j - entry_idx + 1,
                    }

                if target_hit:
                    return {
                        "signal_idx": signal_idx,
                        "entry_idx": entry_idx,
                        "exit_idx": j,
                        "signal_time": signal_row["datetime"],
                        "entry_time": entry_row["datetime"],
                        "exit_time": row["datetime"],
                        "side": side,
                        "signal_name": signal_name,
                        "entry_price": entry_price,
                        "stop_price_initial": stop_price,
                        "be_trigger_price": be_trigger_price,
                        "target_price": target_price,
                        "exit_price": target_price,
                        "exit_reason": "TP",
                        "points_result": config["take_profit_points"],
                        "R_result": config["take_profit_points"] / config["stop_loss_points"],
                        "bars_held": j - entry_idx + 1,
                    }

        else:
            raise ValueError("side must be 'long' or 'short'")

    # Fallback if the trade never exits before data ends.
    final_row = df.iloc[-1]

    if side == "long":
        final_points = float(final_row["close"]) - entry_price
    else:
        final_points = entry_price - float(final_row["close"])

    return {
        "signal_idx": signal_idx,
        "entry_idx": entry_idx,
        "exit_idx": len(df) - 1,
        "signal_time": signal_row["datetime"],
        "entry_time": entry_row["datetime"],
        "exit_time": final_row["datetime"],
        "side": side,
        "signal_name": signal_name,
        "entry_price": entry_price,
        "stop_price_initial": stop_price,
        "be_trigger_price": be_trigger_price,
        "target_price": target_price,
        "exit_price": float(final_row["close"]),
        "exit_reason": "DATA_END",
        "points_result": final_points,
        "R_result": final_points / config["stop_loss_points"],
        "bars_held": len(df) - entry_idx,
    }

In [ ]:
def simulate_fixed_point_trades(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Simulate fixed-point trades from simple green signals.

    Entry is next bar open.
    Only one trade can be open at a time.
    """
    trades = []

    i = 0
    n = len(df)

    while i < n - 1:
        row = df.iloc[i]

        signal_name = row.get("simple_signal_name", "NO_SIGNAL")
        side = row.get("simple_signal_side", "none")

        if signal_name == "NO_SIGNAL" or side == "none":
            i += 1
            continue

        if not is_before_no_new_trades_cutoff(row["datetime"], config):
            i += 1
            continue

        entry_idx = i + 1

        trade = simulate_single_trade(
            df=df,
            entry_idx=entry_idx,
            signal_idx=i,
            side=side,
            signal_name=signal_name,
            config=config,
        )

        trades.append(trade)

        # Enforce one open trade at a time.
        i = trade["exit_idx"] + 1

    return pd.DataFrame(trades)

In [ ]:
trade_log = simulate_fixed_point_trades(signals_df, AUTO_CONFIG)

print(f"Trades simulated: {len(trade_log):,}")

if len(trade_log) > 0:
    display(trade_log.tail(50))
else:
    print("No trades found.")

In [ ]:
if len(trade_log) > 0:
    summary = {
        "total_trades": len(trade_log),
        "wins_TP": int((trade_log["exit_reason"] == "TP").sum()),
        "losses_SL": int((trade_log["exit_reason"] == "SL").sum()),
        "breakevens_BE": int((trade_log["exit_reason"] == "BE").sum()),
        "data_end": int((trade_log["exit_reason"] == "DATA_END").sum()),
        "total_points": trade_log["points_result"].sum(),
        "average_points": trade_log["points_result"].mean(),
        "total_R": trade_log["R_result"].sum(),
        "average_R": trade_log["R_result"].mean(),
        "win_rate_excluding_BE": (
            (trade_log["exit_reason"] == "TP").sum()
            / max(
                1,
                ((trade_log["exit_reason"] == "TP") | (trade_log["exit_reason"] == "SL")).sum(),
            )
        ),
        "average_bars_held": trade_log["bars_held"].mean(),
        "median_bars_held": trade_log["bars_held"].median(),
    }

    display(pd.Series(summary))

In [ ]:
if len(trade_log) > 0:
    exit_breakdown = trade_log["exit_reason"].value_counts(dropna=False)
    side_breakdown = trade_log["side"].value_counts(dropna=False)

    print("Exit breakdown:")
    display(exit_breakdown)

    print("Side breakdown:")
    display(side_breakdown)

In [ ]:
if len(trade_log) > 0:
    trade_log["cumulative_points"] = trade_log["points_result"].cumsum()
    trade_log["cumulative_R"] = trade_log["R_result"].cumsum()

    display(
        trade_log[
            [
                "signal_time",
                "entry_time",
                "exit_time",
                "side",
                "signal_name",
                "entry_price",
                "exit_price",
                "exit_reason",
                "points_result",
                "R_result",
                "bars_held",
                "cumulative_points",
                "cumulative_R",
            ]
        ].tail(50)
    )

## Commit Notes

This commit adds the first fixed-point trade simulation.

Current simulation includes:

- next-bar-open entries
- 29-point stop
- 58-point target
- breakeven after +29 points
- one open trade at a time
- conservative intrabar ambiguity handling
- basic trade log and summary

Not added yet:

- daily stop after 2 consecutive losses
- daily +8% profit cap
- S/A setup tiering
- red-band shift score filters
- trend-death filters
- second-close re-entry logic
- failed-auction continuation logic

## Daily Risk Controls

This section adds daily stop rules to the fixed-point simulator.

Daily controls:

- Stop trading for the day after 2 consecutive losing trades.
- Stop trading for the day after +8% realised daily profit.
- A winning trade resets consecutive losses to 0.
- A losing trade increments consecutive losses by 1.
- A breakeven trade does not increment and does not reset consecutive losses.

Assumption:

Because each trade risks 1%, a -1R trade equals -1%, a +2R trade equals +2%, and a breakeven trade equals 0%.

In [ ]:
def get_session_date(timestamp, config: dict):
    """
    Convert timestamp into the configured session timezone and return the date.
    """
    session_ts = normalise_timestamp_to_session_time(timestamp, config)
    return session_ts.date()


def trade_result_to_account_pct(trade: dict, config: dict) -> float:
    """
    Convert trade R-result into account percentage.

    If risk_per_trade_pct = 1:
    -1R = -1%
    +2R = +2%
    0R = 0%
    """
    return float(trade["R_result"]) * config["risk_per_trade_pct"]

In [ ]:
def simulate_fixed_point_trades_with_daily_controls(
    df: pd.DataFrame,
    config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Simulate fixed-point trades with daily risk controls.

    Includes:
    - next-bar-open entries
    - 29-point stop
    - 58-point target
    - breakeven after +29 points
    - one open trade at a time
    - no new trades after configured cutoff
    - daily stop after max consecutive losses
    - daily stop after daily profit cap
    """
    trades = []
    skipped_signals = []

    current_session_date = None
    daily_realized_pct = 0.0
    consecutive_losses = 0

    i = 0
    n = len(df)

    while i < n - 1:
        row = df.iloc[i]

        signal_name = row.get("simple_signal_name", "NO_SIGNAL")
        side = row.get("simple_signal_side", "none")

        if signal_name == "NO_SIGNAL" or side == "none":
            i += 1
            continue

        signal_date = get_session_date(row["datetime"], config)

        # Reset daily counters when a new session date begins
        if signal_date != current_session_date:
            current_session_date = signal_date
            daily_realized_pct = 0.0
            consecutive_losses = 0

        # Session cutoff
        if not is_before_no_new_trades_cutoff(row["datetime"], config):
            skipped_signals.append(
                {
                    "signal_idx": i,
                    "signal_time": row["datetime"],
                    "session_date": signal_date,
                    "signal_name": signal_name,
                    "side": side,
                    "skip_reason": "SESSION_CUTOFF",
                    "daily_realized_pct": daily_realized_pct,
                    "consecutive_losses": consecutive_losses,
                }
            )
            i += 1
            continue

        # Daily consecutive-loss stop
        if consecutive_losses >= config["max_consecutive_losses"]:
            skipped_signals.append(
                {
                    "signal_idx": i,
                    "signal_time": row["datetime"],
                    "session_date": signal_date,
                    "signal_name": signal_name,
                    "side": side,
                    "skip_reason": "MAX_CONSECUTIVE_LOSSES",
                    "daily_realized_pct": daily_realized_pct,
                    "consecutive_losses": consecutive_losses,
                }
            )
            i += 1
            continue

        # Daily profit cap
        if daily_realized_pct >= config["daily_profit_cap_pct"]:
            skipped_signals.append(
                {
                    "signal_idx": i,
                    "signal_time": row["datetime"],
                    "session_date": signal_date,
                    "signal_name": signal_name,
                    "side": side,
                    "skip_reason": "DAILY_PROFIT_CAP",
                    "daily_realized_pct": daily_realized_pct,
                    "consecutive_losses": consecutive_losses,
                }
            )
            i += 1
            continue

        entry_idx = i + 1

        trade = simulate_single_trade(
            df=df,
            entry_idx=entry_idx,
            signal_idx=i,
            side=side,
            signal_name=signal_name,
            config=config,
        )

        trade_pct_result = trade_result_to_account_pct(trade, config)

        trade["session_date"] = signal_date
        trade["daily_realized_pct_before"] = daily_realized_pct
        trade["consecutive_losses_before"] = consecutive_losses
        trade["account_pct_result"] = trade_pct_result

        # Update daily realised PnL
        daily_realized_pct += trade_pct_result

        # Update consecutive-loss counter
        if trade["points_result"] < 0:
            consecutive_losses += 1
        elif trade["points_result"] > 0:
            consecutive_losses = 0
        else:
            # Breakeven is neutral
            consecutive_losses = consecutive_losses

        trade["daily_realized_pct_after"] = daily_realized_pct
        trade["consecutive_losses_after"] = consecutive_losses
        trade["daily_stop_active_after"] = (
            consecutive_losses >= config["max_consecutive_losses"]
            or daily_realized_pct >= config["daily_profit_cap_pct"]
        )

        trades.append(trade)

        # Enforce one open trade at a time
        i = trade["exit_idx"] + 1

    return pd.DataFrame(trades), pd.DataFrame(skipped_signals)

In [ ]:
controlled_trade_log, skipped_signal_log = simulate_fixed_point_trades_with_daily_controls(
    signals_df,
    AUTO_CONFIG,
)

print(f"Controlled trades simulated: {len(controlled_trade_log):,}")
print(f"Skipped signals: {len(skipped_signal_log):,}")

if len(controlled_trade_log) > 0:
    display(controlled_trade_log.tail(50))

if len(skipped_signal_log) > 0:
    display(skipped_signal_log.tail(50))

In [ ]:
if len(controlled_trade_log) > 0:
    controlled_summary = {
        "total_trades": len(controlled_trade_log),
        "wins_TP": int((controlled_trade_log["exit_reason"] == "TP").sum()),
        "losses_SL": int((controlled_trade_log["exit_reason"] == "SL").sum()),
        "breakevens_BE": int((controlled_trade_log["exit_reason"] == "BE").sum()),
        "data_end": int((controlled_trade_log["exit_reason"] == "DATA_END").sum()),
        "total_points": controlled_trade_log["points_result"].sum(),
        "average_points": controlled_trade_log["points_result"].mean(),
        "total_R": controlled_trade_log["R_result"].sum(),
        "average_R": controlled_trade_log["R_result"].mean(),
        "total_account_pct": controlled_trade_log["account_pct_result"].sum(),
        "average_account_pct": controlled_trade_log["account_pct_result"].mean(),
        "win_rate_excluding_BE": (
            (controlled_trade_log["exit_reason"] == "TP").sum()
            / max(
                1,
                (
                    (controlled_trade_log["exit_reason"] == "TP")
                    | (controlled_trade_log["exit_reason"] == "SL")
                ).sum(),
            )
        ),
        "average_bars_held": controlled_trade_log["bars_held"].mean(),
        "median_bars_held": controlled_trade_log["bars_held"].median(),
    }

    display(pd.Series(controlled_summary))

In [ ]:
if len(controlled_trade_log) > 0:
    daily_summary = (
        controlled_trade_log
        .groupby("session_date")
        .agg(
            trades=("points_result", "count"),
            total_points=("points_result", "sum"),
            total_R=("R_result", "sum"),
            total_account_pct=("account_pct_result", "sum"),
            wins=("exit_reason", lambda x: (x == "TP").sum()),
            losses=("exit_reason", lambda x: (x == "SL").sum()),
            breakevens=("exit_reason", lambda x: (x == "BE").sum()),
            max_consecutive_losses_after=("consecutive_losses_after", "max"),
            daily_stop_triggered=("daily_stop_active_after", "max"),
        )
        .reset_index()
    )

    display(daily_summary.tail(30))

In [ ]:
if len(controlled_trade_log) > 0:
    controlled_trade_log["cumulative_points"] = controlled_trade_log["points_result"].cumsum()
    controlled_trade_log["cumulative_R"] = controlled_trade_log["R_result"].cumsum()
    controlled_trade_log["cumulative_account_pct"] = controlled_trade_log["account_pct_result"].cumsum()

    display(
        controlled_trade_log[
            [
                "signal_time",
                "entry_time",
                "exit_time",
                "session_date",
                "side",
                "signal_name",
                "entry_price",
                "exit_price",
                "exit_reason",
                "points_result",
                "R_result",
                "account_pct_result",
                "daily_realized_pct_before",
                "daily_realized_pct_after",
                "consecutive_losses_before",
                "consecutive_losses_after",
                "daily_stop_active_after",
                "cumulative_points",
                "cumulative_R",
                "cumulative_account_pct",
            ]
        ].tail(50)
    )

In [ ]:
if len(skipped_signal_log) > 0:
    print("Skipped signal reasons:")
    display(skipped_signal_log["skip_reason"].value_counts())
else:
    print("No signals skipped by daily/session controls.")

## Commit Notes

This commit adds daily risk controls to the fixed-point simulator.

Added:

- stop after 2 consecutive losses
- stop after +8% realised daily profit
- win resets consecutive losses
- loss increments consecutive losses
- breakeven is neutral
- skipped signal log
- daily summary table

Not added yet:

- S/A setup tiering
- red-band shift score filter
- trend-death filter
- second-close re-entry logic
- failed-auction continuation logic

## Trade Diagnostics

This section analyses the simulated trade log before adding more filters.

The goal is to understand which conditions are helping or hurting:

- long vs short
- time of day
- red-band shift strength
- candle body quality
- distance from green at entry signal
- exit type

No strategy rules are changed in this section.

In [ ]:
def attach_signal_context_to_trades(
    trades: pd.DataFrame,
    signal_df: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """
    Attach signal-candle context to each simulated trade.

    This helps diagnose which signal features are associated with better or worse outcomes.
    """
    if trades.empty:
        return trades.copy()

    context_cols = [
        "datetime",
        "close",
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
        "body_ratio",
        "vwap_shift",
        "upper_green_shift",
        "upper_orange_shift",
        "upper_red_shift",
        "lower_green_shift",
        "lower_orange_shift",
        "lower_red_shift",
        "bullish_red_shift_strength",
        "bearish_red_shift_strength",
        "bullish_red_shift_label",
        "bearish_red_shift_label",
        "accepted_above_vwap",
        "accepted_below_vwap",
        "possible_chop",
        "long_close_through_green_points",
        "long_extension_from_green_points",
        "short_close_through_green_points",
        "short_extension_from_green_points",
    ]

    available_context_cols = [col for col in context_cols if col in signal_df.columns]

    signal_context = (
        signal_df[available_context_cols]
        .copy()
        .reset_index()
        .rename(columns={"index": "signal_idx"})
    )

    enriched = trades.merge(
        signal_context,
        on="signal_idx",
        how="left",
        suffixes=("", "_signal"),
    )

    enriched["signal_time_london"] = enriched["signal_time"].apply(
        lambda ts: normalise_timestamp_to_session_time(ts, config)
    )

    enriched["signal_hour_london"] = enriched["signal_time_london"].dt.hour

    enriched["signal_time_bucket"] = pd.cut(
        enriched["signal_hour_london"],
        bins=[0, 12, 14, 16, 18, 20, 24],
        labels=[
            "00-12",
            "12-14",
            "14-16",
            "16-18",
            "18-20",
            "20-24",
        ],
        right=False,
    )

    enriched["trade_red_shift_strength"] = np.where(
        enriched["side"] == "long",
        enriched["bullish_red_shift_strength"],
        enriched["bearish_red_shift_strength"],
    )

    enriched["trade_red_shift_label"] = np.where(
        enriched["side"] == "long",
        enriched["bullish_red_shift_label"],
        enriched["bearish_red_shift_label"],
    )

    enriched["trade_close_through_green_points"] = np.where(
        enriched["side"] == "long",
        enriched["long_close_through_green_points"],
        enriched["short_close_through_green_points"],
    )

    enriched["trade_extension_from_green_points"] = np.where(
        enriched["side"] == "long",
        enriched["long_extension_from_green_points"],
        enriched["short_extension_from_green_points"],
    )

    enriched["body_ratio_bucket"] = pd.cut(
        enriched["body_ratio"],
        bins=[0.0, 0.25, 0.50, 0.75, 1.00],
        labels=[
            "0.00-0.25",
            "0.25-0.50",
            "0.50-0.75",
            "0.75-1.00",
        ],
        include_lowest=True,
    )

    enriched["extension_bucket"] = pd.cut(
        enriched["trade_extension_from_green_points"],
        bins=[-np.inf, 1, 2, 4, 6, 8, np.inf],
        labels=[
            "<=1",
            "1-2",
            "2-4",
            "4-6",
            "6-8",
            ">8",
        ],
    )

    return enriched

In [ ]:
diagnostic_trade_log = attach_signal_context_to_trades(
    controlled_trade_log,
    signals_df,
    AUTO_CONFIG,
)

print(f"Diagnostic trade log rows: {len(diagnostic_trade_log):,}")

if len(diagnostic_trade_log) > 0:
    display(
        diagnostic_trade_log[
            [
                "signal_time",
                "signal_time_london",
                "side",
                "signal_name",
                "exit_reason",
                "points_result",
                "R_result",
                "account_pct_result",
                "body_ratio",
                "body_ratio_bucket",
                "trade_red_shift_strength",
                "trade_red_shift_label",
                "trade_close_through_green_points",
                "trade_extension_from_green_points",
                "extension_bucket",
                "signal_hour_london",
                "signal_time_bucket",
            ]
        ].tail(50)
    )

In [ ]:
def performance_summary_by(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Summarise trade performance by one grouping column.
    """
    if df.empty:
        return pd.DataFrame()

    summary = (
        df.groupby(group_col, dropna=False)
        .agg(
            trades=("R_result", "count"),
            wins=("exit_reason", lambda x: (x == "TP").sum()),
            losses=("exit_reason", lambda x: (x == "SL").sum()),
            breakevens=("exit_reason", lambda x: (x == "BE").sum()),
            total_points=("points_result", "sum"),
            avg_points=("points_result", "mean"),
            total_R=("R_result", "sum"),
            avg_R=("R_result", "mean"),
            total_account_pct=("account_pct_result", "sum"),
            avg_bars_held=("bars_held", "mean"),
        )
        .reset_index()
    )

    summary["win_rate_ex_BE"] = summary["wins"] / (
        summary["wins"] + summary["losses"]
    ).replace(0, np.nan)

    return summary.sort_values("total_R", ascending=False)

In [ ]:
if len(diagnostic_trade_log) > 0:
    print("Performance by side:")
    display(performance_summary_by(diagnostic_trade_log, "side"))

    print("Performance by signal name:")
    display(performance_summary_by(diagnostic_trade_log, "signal_name"))

    print("Performance by exit reason:")
    display(performance_summary_by(diagnostic_trade_log, "exit_reason"))

In [ ]:
if len(diagnostic_trade_log) > 0:
    print("Performance by London hour:")
    display(performance_summary_by(diagnostic_trade_log, "signal_hour_london"))

    print("Performance by London time bucket:")
    display(performance_summary_by(diagnostic_trade_log, "signal_time_bucket"))

In [ ]:
if len(diagnostic_trade_log) > 0:
    print("Performance by red-band shift label:")
    display(performance_summary_by(diagnostic_trade_log, "trade_red_shift_label"))

    print("Performance by body-ratio bucket:")
    display(performance_summary_by(diagnostic_trade_log, "body_ratio_bucket"))

    print("Performance by extension-from-green bucket:")
    display(performance_summary_by(diagnostic_trade_log, "extension_bucket"))

In [ ]:
if len(diagnostic_trade_log) > 0:
    diagnostic_trade_log["cumulative_R"] = diagnostic_trade_log["R_result"].cumsum()
    diagnostic_trade_log["cumulative_points"] = diagnostic_trade_log["points_result"].cumsum()
    diagnostic_trade_log["cumulative_account_pct"] = diagnostic_trade_log["account_pct_result"].cumsum()

    display(
        diagnostic_trade_log[
            [
                "signal_time",
                "side",
                "signal_name",
                "exit_reason",
                "points_result",
                "R_result",
                "cumulative_R",
                "cumulative_points",
                "cumulative_account_pct",
            ]
        ].tail(50)
    )

In [ ]:
import matplotlib.pyplot as plt

if len(diagnostic_trade_log) > 0:
    plt.figure(figsize=(10, 5))
    plt.plot(diagnostic_trade_log["signal_time"], diagnostic_trade_log["cumulative_R"])
    plt.title("Cumulative R — Simple Green Signal Strategy")
    plt.xlabel("Signal time")
    plt.ylabel("Cumulative R")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Commit Notes

This commit adds diagnostics only.

No trading rules were changed.

The next strategy-improvement commit should be chosen based on these diagnostics.

Possible next filters:

- red-band shift strength filter
- time-of-day filter
- body-ratio filter
- extension-from-green tightening
- trend-death filter